# Figure-in-noise: the versions I tried

Meysam Amirsardari

Three task designs, the checks on each, and what one listener actually did.

[Open in Colab](https://colab.research.google.com/github/MeysamAmirsardari/SeqSFG_task/blob/main/notebooks/SeqSFG_overview.ipynb)

Run all. Headphones, fixed volume. Audio does not autoplay.

Pinned to commit `f53f70e51552`. The pilot data is in the repo and was recorded with exactly the
configuration you will hear.

In [ ]:
#@title setup
import sys, subprocess, importlib.util, json
from pathlib import Path
SOURCE_REF='f53f70e515520e9f00254773fd6969f41decae3f'
REPO='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
IN_COLAB=bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'))
if IN_COLAB:
    ROOT=Path('/content')/('seqsfg-'+SOURCE_REF[:12])
    if not ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO,str(ROOT)],check=True)
        subprocess.run(['git','-C',str(ROOT),'checkout','--detach',SOURCE_REF],check=True)
else:
    ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'seqsfg/stimulus.py').exists()),None)
for m in ['numpy','scipy','matplotlib']:
    if importlib.util.find_spec(m) is None: subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
sys.path.insert(0,str(ROOT))

import csv, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Audio, Markdown
from seqsfg.config import Config, validate
from seqsfg.stimulus import make_trial, render_interval, FIGURE, BACKGROUND
from seqsfg import yesno, exposure as X

cfg=Config.from_dict(json.loads((ROOT/'pilot_config.json').read_text()))
D=validate(cfg); SR=cfg.sample_rate; OCT=np.log2(D.channel_freqs_hz)
plt.rcParams.update({'figure.dpi':115,'axes.spines.top':False,'axes.spines.right':False,
                      'axes.grid':True,'grid.alpha':.25,'font.size':9})
C={'fig':'#c1272d','other':'#1b6ca8','bg':'#c3c8cf','ok':'#2e7d32','no':'#b0481f'}
TICKS=[250,1000,4000]

def play(x):
    pad=np.zeros(int(.05*SR),dtype=np.float32)
    return Audio(np.clip(np.concatenate([pad,x]),-1,1),rate=SR,normalize=False)

def part(iv,kind):
    m=iv.kind==kind; out=iv.copy()
    for a in ('onset','channel','phase','kind','element','component'): setattr(out,a,getattr(iv,a)[m])
    return render_interval(cfg,out,D)

def raster(ax,iv,col,t0=0,t1=None,title=''):
    t=iv.onset*cfg.grid_ms/1000.; y=OCT[iv.channel]; m=iv.kind==FIGURE
    ax.scatter(t[~m],y[~m],s=5,c=C['bg'],marker='s',lw=0)
    ax.scatter(t[m],y[m],s=22,c=col,marker='s',lw=0)
    ax.set_xlim(t0, t1 or cfg.interval_dur_ms/1000.); ax.set_xlabel('time (s)')
    ax.set_yticks(np.log2(TICKS)); ax.set_yticklabels([str(v) for v in TICKS])
    ax.set_title(title,fontsize=10)

SUM=json.loads((ROOT/'verification/pilot_summary.json').read_text())['sessions']
def cells(sess):
    return SUM[sess]['cells']

def dprime(h,ns,f,nn):
    return float(stats.norm.ppf((h+.5)/(ns+1))-stats.norm.ppf((f+.5)/(nn+1)))

print(f"{D.n_channels} channels, {cfg.pool_low_hz:.0f}-{D.channel_freqs_hz[-1]:.0f} Hz")
print(f"{cfg.n_elements} elements of {cfg.n_components} tones, {cfg.tone_dur_ms:.0f} ms each, "
      f"{1000/cfg.iei_max_ms:.1f}-{1000/cfg.iei_min_ms:.1f} per second, {cfg.interval_dur_ms/1000:g} s")
print('config', cfg.hash())

---
## 1 · The sound

A cloud of random tones. Somewhere in it, seven tones start together and come back on the same
pitches, nine times.

In [ ]:
tr=make_trial(cfg,101,0.0,'rising',d=D)
iv=tr.recurring
display(Markdown('cloud only')); display(play(part(iv,BACKGROUND)))
display(Markdown('figure only')); display(play(part(iv,FIGURE)))
display(Markdown('both, the real stimulus')); display(play(render_interval(cfg,iv,D)))

fig,ax=plt.subplots(figsize=(11,3.2)); raster(ax,iv,C['fig'],title='red = the seven tones that start together')
ax.set_ylabel('Hz'); plt.tight_layout(); plt.show()

---
## 2 · Task A, two intervals

Two sounds. Both have a group every element. In one the group keeps the same pitches. In the
other it moves to new pitches each time.

Which one repeats?

In [ ]:
display(Markdown('interval A')); display(play(render_interval(cfg,tr.recurring,D)))
display(Markdown('interval B')); display(play(render_interval(cfg,tr.other,D)))
fig,axes=plt.subplots(1,2,figsize=(12,3.4),sharey=True)
raster(axes[0],tr.recurring,C['fig'],title='A: same pitches every time')
raster(axes[1],tr.other,C['other'],title='B: new pitches every time')
axes[0].set_ylabel('Hz'); plt.tight_layout(); plt.show()

---
## 3 · The knob

Inside an element the seven tones can start together, or spread out. `step` is the gap between
neighbours. Bigger step, less overlap.

In [ ]:
steps=list(cfg.steps_ms)
ov=[max(0,1-s/cfg.tone_dur_ms) for s in steps]
fig,ax=plt.subplots(figsize=(5.6,2.8))
ax.plot(steps,ov,'o-',color='k',lw=1.2); ax.set_xlabel('step (ms)'); ax.set_ylabel('overlap')
ax.set_ylim(0,1.05); plt.tight_layout(); plt.show()
for s in (0.0,17.0):
    display(Markdown(f'step {s:.0f} ms, overlap {max(0,1-s/cfg.tone_dur_ms):.2f}'))
    display(play(render_interval(cfg,make_trial(cfg,7,s,'rising',d=D).recurring,D)))
fig,axes=plt.subplots(1,2,figsize=(12,3.2),sharey=True)
for ax,s in zip(axes,(0.0,17.0)):
    raster(ax,make_trial(cfg,7,s,'rising',d=D).recurring,C['fig'],t0=.3,t1=1.6,title=f'step {s:.0f} ms')
axes[0].set_ylabel('Hz'); plt.tight_layout(); plt.show()

---
## 4 · Task A, one listener

P01, 120 trials, 10 per cell.

In [ ]:
cc=cells('L1/session_06')
fig,ax=plt.subplots(figsize=(6.4,3.6))
for v,col in (('rising',C['fig']),('redrawn',C['other'])):
    xs,ys,lo,hi=[],[],[],[]
    for key in sorted([k for k in cc if k.startswith(v+'|')],key=lambda k:float(k.split('|')[1])):
        n,k_=cc[key]['n'],cc[key]['n_correct']; p=k_/n
        a,b=stats.beta.interval(.95,k_+.5,n-k_+.5)
        xs.append(float(key.split('|')[1])); ys.append(p); lo.append(max(0,p-a)); hi.append(max(0,b-p))
    ax.errorbar(xs,ys,yerr=[lo,hi],fmt='o-',color=col,capsize=3,lw=1.4,label=v)
ax.axhline(.5,color='k',lw=.8,ls='--'); ax.text(17,.52,'chance',fontsize=8,ha='right')
ax.set_xlabel('step (ms)'); ax.set_ylabel('correct'); ax.set_ylim(0,1.05); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

Both curves: perfect when the tones start together, chance as soon as they do not.

So the task measures whether the tones are simultaneous, not whether the pitches repeat.

The two orders sit on top of each other. No sign that the order matters.

---
## 5 · The two orders I compared

Same pitches, different timing inside the element.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,3.2),sharey=True)
for ax,(v,ttl) in zip(axes,[('rising','rising: low to high, same every element'),
                            ('redrawn','redrawn: new order every element')]):
    raster(ax,make_trial(cfg,55,14.0,v,d=D).recurring,C['fig'],t0=.3,t1=1.6,title=ttl)
axes[0].set_ylabel('Hz'); plt.tight_layout(); plt.show()
for v in ('rising','redrawn'):
    display(Markdown(f'{v}, step 14 ms')); display(play(render_interval(cfg,make_trial(cfg,55,14.0,v,d=D).recurring,D)))

---
## 6 · Task B, one interval

One sound. Was a figure there, yes or no. Easier to follow than comparing two.

In [ ]:
for present in (True,False):
    display(Markdown(f'**{"yes" if present else "no"}**'))
    display(play(render_interval(cfg,yesno.build_interval(cfg,D,31 if present else 32,7.0,present,'roving'),D)))

---
## 7 · What "no figure" should be

Three options. I checked whether a machine that never hears a group can separate them from
"yes" trials, using 65 acoustic measures.

In [ ]:
for k,lab in [('plain','plain cloud, no group at all'),
              ('scattered','same pitches, never lined up'),
              ('roving','a group, but new pitches each time')]:
    display(Markdown(f'`{k}` — {lab}')); display(play(render_interval(cfg,yesno.build_interval(cfg,D,88,7.0,False,k),D)))

txt=(ROOT/'verification/yesno_roving_report.txt').read_text()
res={}
for k in ('roving','scattered','plain'):
    t=(ROOT/f'verification/yesno_{k}_report.txt').read_text()
    pc=float([l for l in t.splitlines() if 'correct' in l and 'hit rate' in l][0].split()[-2].rstrip('%'))
    res[k]=pc
fig,ax=plt.subplots(figsize=(5.4,2.9))
ax.bar(list(res),[res[k] for k in res],color=[C['ok'] if k=='roving' else C['no'] for k in res])
ax.axhline(50,color='k',lw=.8,ls='--'); ax.set_ylabel('% correct, machine'); ax.set_ylim(0,100)
for i,k in enumerate(res): ax.text(i,res[k]+2,f'{res[k]:.0f}%',ha='center',fontsize=9)
plt.tight_layout(); plt.show()

A plain cloud is 90% solvable without hearing anything. The envelope gives it away: a chord
every 316 ms is a rhythm.

So "no" means a group that does not repeat.

---
## 8 · Task B, one listener

P01, 120 trials, 20 per step.

In [ ]:
c7=cells('L1/session_07'); xs,ys=[],[]
for key in sorted(c7,key=lambda k:float(k.rstrip('|'))):
    v=c7[key]; xs.append(float(key.rstrip('|')))
    ys.append(dprime(v['hits'],v['n_signal'],v['false_alarms'],v['n_noise']))
fig,ax=plt.subplots(figsize=(6.4,3.4))
ax.plot(xs,ys,'o-',color=C['fig'],lw=1.4)
ax.axhline(0,color='k',lw=.8,ls='--'); ax.set_xlabel('step (ms)'); ax.set_ylabel("d'")
plt.tight_layout(); plt.show()
print('10 present and 10 absent per point')

Same shape. Something at step 0, nothing after.

---
## 9 · Task C, training inside the task

Two orders, P and Q, on the same seven pitches. Chosen to share no transition.

Test both, then give extra trials of one only, then test both again.

In [ ]:
P,Q,OV=X.choose_orders(cfg.n_components,20260910)
S=X.figure_set_for(cfg,D,X.ExposureConfig())
print('P heard order',X.heard_order(P),'   Q heard order',X.heard_order(Q))
print('shared transitions',OV['shared_directed_transitions'],'of',OV['n_directed_transitions'],
      '  same slot',OV['components_in_the_same_slot'])
fig,axes=plt.subplots(1,2,figsize=(12,3.2),sharey=True)
for ax,(k,o) in zip(axes,[('P',P),('Q',Q)]):
    raster(ax,X.build_trial(cfg,D,S,o,64,14.0,True),C['fig'],t0=.3,t1=1.6,title=f'order {k}')
axes[0].set_ylabel('Hz'); plt.tight_layout(); plt.show()
for k,o in (('P',P),('Q',Q)):
    display(Markdown(f'order {k}')); display(play(render_interval(cfg,X.build_trial(cfg,D,S,o,64,14.0,True),D)))

The effect:

D = (post trained − pre trained) − (post untrained − pre untrained)

Subtracting the untrained side removes plain practice.

In [ ]:
c8=cells('L1/session_08')
dd={}
fig,ax=plt.subplots(figsize=(6.8,3.6))
for role,col in (('trained',C['fig']),('untrained',C['other'])):
    for phase,ls,mk in (('pre','--','o'),('post','-','s')):
        xs,ys=[],[]
        for key in sorted([k for k in c8 if k.startswith(f'{phase}|{role}|')],
                          key=lambda k:float(k.split('|')[2])):
            v=c8[key]; st=float(key.split('|')[2])
            g=dprime(v['hits'],v['n_signal'],v['false_alarms'],v['n_noise'])
            dd[(phase,role,st)]=g; xs.append(st); ys.append(g)
        ax.plot(xs,ys,ls,marker=mk,color=col,lw=1.3,ms=5,label=f'{role} {phase}')
ax.axhline(0,color='k',lw=.8,ls='--'); ax.set_xlabel('step (ms)'); ax.set_ylabel("d'")
ax.legend(frameon=False,fontsize=8,ncol=2); plt.tight_layout(); plt.show()
for st in sorted({k[2] for k in dd if k[2]>0}):
    Dv=((dd[('post','trained',st)]-dd[('pre','trained',st)])
        -(dd[('post','untrained',st)]-dd[('pre','untrained',st)]))
    print(f"  {st:>4.0f} ms   D = {Dv:+.2f}")
print('  8 trials per cell, so each of these is worth about +-2')

D is flat and the intervals are very wide. One listener, 8 trials a cell. Not enough to say
anything.

---
## 10 · The check I run on every version

Ideal observers that get one property of the sound and nothing else. They have to sit at chance,
or the task can be done without hearing a figure.

In [ ]:
t=(ROOT/'verification/pilot_battery_report.txt').read_text().splitlines()
i=[n for n,l in enumerate(t) if "ladder 'rising'" in l][0]
import re
names,vals=[],[]
for l in t[i+1:i+6]:
    m=re.match(r"\s*(.+?)\s+d' = ([+-][0-9.]+)",l)
    if m: names.append(m.group(1)); vals.append(float(m.group(2)))
fig,ax=plt.subplots(figsize=(6.6,2.9))
ax.barh(names[::-1],vals[::-1],color=C['bg'],edgecolor='k',lw=.6)
ax.axvline(0,color='k',lw=.8); ax.set_xlim(-1,1); ax.set_xlabel("d' (0 = chance)")
plt.tight_layout(); plt.show()
print([l for l in t[i:i+12] if 'global permutation' in l][0].strip())

---
## 11 · Where it is

Works: the stimulus is controlled, the machine checks pass, three tasks run end to end.

Does not work yet: past step 0 the listener is at chance, so there is no curve to fit and no
order effect to find.

Next: more trials per point, and easier settings so the curve starts above chance.